# Step 1: Install the required packages/libraries 

In [ ]:
# Step 1: Install the required packages/libraries 

# pip install pandas
# pip install openpyxl

# Step 2: Import required libraries

In [ ]:
# Import Libraries

import sqlite3
import pandas as pd


# Step 3: Load/Import data & Build connection

In [3]:
# Load Excel file
df = pd.read_excel(r"customer_sample_500.csv.xlsx", dtype=str)

# Replace NaN with empty string
df = df.fillna("")

# Connect to SQLite
conn = sqlite3.connect(":memory:")

# Load dataframe into SQLite table
df.to_sql("customers", conn, index=False, if_exists="replace")

# Create cursor
cursor = conn.cursor()

print("Data successfully loaded into SQLite table: customers")

Data successfully loaded into SQLite table: customers


# Step 4: Data Quality | Completeness Check

## Datasets using: 
- customer_sample_500


In [6]:
#-----Completeness check------

def completeness_check(cursor, df, output_file="completeness_results.xlsx"):

    results = []

    # Loop through each column
    for col in df.columns:
        
        query = f"""
        SELECT COUNT(*)
        FROM customers
        WHERE {col} IS NULL OR TRIM({col}) = ''
        """

        # Execute query
        missing = cursor.execute(query).fetchone()[0]

        results.append({
            "Column": col,
            "Missing Values": missing
        })

    # Convert results to DataFrame
    result_df = pd.DataFrame(results)

    # Save output
    result_df.to_excel(output_file, index=False)

    print("Output saved to", output_file)

    return result_df


# Run completeness check
result = completeness_check(cursor, df)


Output saved to completeness_results.xlsx


In [ ]:
'''Uniqueness_check'''

# -------- Step 1: Detect columns that contain only unique values --------

def find_unique_columns_sql(cursor, df):

    unique_columns = []

    for col in df.columns:

        query = f"""
        SELECT COUNT(*) = COUNT(DISTINCT {col})
        FROM customers
        """

        result = cursor.execute(query).fetchone()[0]

        if result == 1:
            unique_columns.append(col)

    return unique_columns


# Run Step 1
unique_cols = find_unique_columns_sql(cursor, df)

# Save detected unique columns
unique_df = pd.DataFrame(unique_cols, columns=["Unique_Columns"])
unique_df.to_excel("unique_columns.xlsx", index=False)

print("Step 1 completed: unique_columns.xlsx created")


# -------- Step 2: Check duplicate count in those columns --------

duplicate_results = []

for col in unique_cols:

    query = f"""
    SELECT COUNT(*) - COUNT(DISTINCT {col})
    FROM customers
    """

    duplicates = cursor.execute(query).fetchone()[0]

    duplicate_results.append({
        "Column": col,
        "Duplicate_Count": duplicates
    })


# Convert results to DataFrame
duplicate_df = pd.DataFrame(duplicate_results)

# Save results
duplicate_df.to_excel("uniqueness_results.xlsx", index=False)

print("Step 2 completed: uniqueness_results.xlsx created")

Step 1 completed: unique_columns.xlsx created
Step 2 completed: uniqueness_results.xlsx created


In [ ]:
#------ Validity_check------

def run_validity_checks(cursor):
    '''
    Add docstring
    '''

    validity_rules = {
        "Invalid_SIRET": """
            SELECT COUNT(*) FROM customers
            WHERE LENGTH(SIRET) != 14
        """,

        "Invalid_SIREN": """
            SELECT COUNT(*) FROM customers
            WHERE LENGTH(SIREN) != 9
        """,

        "Invalid_Country_Code": """
            SELECT COUNT(*) FROM customers
            WHERE LENGTH(COUNTRY_CD) != 2
        """,

        "Invalid_SAP_Postal_Code": """
            SELECT COUNT(*) FROM customers
            WHERE SAP_POSTAL_ZIP_CD GLOB '*[^0-9]*'
        """,

        "Invalid_LUCI_Postal_Code": """
            SELECT COUNT(*) FROM customers
            WHERE LUCI_POSTAL_ZIP_CD GLOB '*[^0-9]*'
        """,

        "Invalid_Customer_Delivery_Block": """
            SELECT COUNT(*) FROM customers
            WHERE CUSTOMER_DELIVERY_BLOC NOT IN ('00','01')
        """,

        "Invalid_GERS": """
            SELECT COUNT(*) FROM customers
            WHERE GERS GLOB '*[^0-9]*'
        """
    }

    delivery_columns = [
        "CUSTOMER_ORDER_BLOCK",
        "CUSTOMER_SALES_DELIVERY_BLOC",
        "CUSTOMER_SALES_ORDER_BLOCK",
        "DELETION_BLOCK"
    ]

    for col in delivery_columns:
        validity_rules[f"{col}_Invalid"] = f"""
            SELECT COUNT(*) FROM customers
            WHERE {col} NOT IN ('00','01')
        """

    results = []

    for rule, query in validity_rules.items():
        invalid_count = cursor.execute(query).fetchone()[0]
        results.append({
            "Check": rule,
            "Invalid_Count": invalid_count
        })

    validity_df = pd.DataFrame(results)

    file_name = "Validity_results.xlsx"

    validity_df.to_excel(file_name, sheet_name="Validity_Check", index=False)

    print(f"Output saved in file: {file_name}")


# run validity checks
run_validity_checks(cursor)

NameError: name 'cursor' is not defined

In [ ]:
#------Consistency check -------

def run_consistency_checks(cursor):

    consistency_rules = {
        "Country_Mismatch": """
            SELECT COUNT(*)
            FROM customers
            WHERE COUNTRY_CD != LUCI_COUNTRY_CD
        """
    }

    results = []

    for rule, query in consistency_rules.items():
        mismatch_count = cursor.execute(query).fetchone()[0]

        results.append({
            "Check": rule,
            "Mismatch_Count": mismatch_count
        })

    consistency_df = pd.DataFrame(results)

    file_name = "Consistency_results.xlsx"

    consistency_df.to_excel(file_name, sheet_name="Consistency_Check", index=False)

    print(f"Output saved in file: {file_name}")


# run consistency check
run_consistency_checks(cursor)

Output saved in file: Consistency_output.xlsx
